모델은 너무 과소적합되지도 않으면서 너무 과적합되지도 않는 게 좋은데요. 우선 과소적합을 막는 것부터 생각해 볼게요. 과소적합은 모델이 너무 단순해서 데이터의 패턴을 제대로 학습하지 못하는 상황을 뜻해요. 그래서 과소적합을 방지하려면 충분히 복잡한 모델을 사용하는 게 좋습니다. 회귀 문제에서는 선형 회귀보다 높은 차수의 다항 회귀를 사용하는 방법이 있어요.
admission.csv

In [29]:
PATH='data/admission.csv'

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, classification_report , confusion_matrix


admission_df = pd.read_csv(PATH)
admission_df.head()

X = admission_df.drop('Chance of Admit', axis=1)
y = admission_df['Chance of Admit']    

poly = PolynomialFeatures(6)
X_poly = poly.fit_transform(X)
print("Polynomial Features shape:", X_poly.shape)
print(X_poly[:5])

Polynomial Features shape: (500, 3003)
[[  1.       1.     337.     ...  93.1225   9.65     1.    ]
 [  1.       2.     324.     ...  78.6769   8.87     1.    ]
 [  1.       3.     316.     ...  64.       8.       1.    ]
 [  1.       4.     322.     ...  75.1689   8.67     1.    ]
 [  1.       5.     314.     ...   0.       0.       0.    ]]


In [ ]:


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## 비 정규화 모델
model = LinearRegression()
model.fit(X_train, y_train) ##학습

train_pred = model.predict(X_train)## 학습기준 예측
test_pred = model.predict(X_test) ## 시험평가 예측
 
print("Polymomial Regression (Non-Normalized) Results:)")
print("Train MSE:", mean_squared_error(y_train, train_pred))
print("Test MSE:", mean_squared_error(y_test, test_pred))
# print("Classification Report for train:")
# print(classification_report(y_train, y_train_class))
# print("Classification Report for test:")
# print(classification_report(y_test, y_test_class))


Polymomial Regression (Non-Normalized) Results:)
Train MSE: 0.003375510480221529
Test MSE: 0.003500684226487506


이처럼 6차 다항 회귀 모델을 사용하면 training MSE는 매우 작은데 test MSE는 상대적으로 크게 나오는 경우가 많아요. 모델이 training 데이터에 지나치게 잘 맞춰져 과적합이 발생한 상황이라고 볼 수 있죠.

이제 정규화를 적용해서 과적합을 줄여 보겠습니다. 먼저 L1 정규화(Lasso)부터 살펴볼게요. 다항 회귀와 정규화 모델은 피처 스케일에 민감하기 때문에, 먼저 표준화를 해 주겠습니다.

In [26]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


L1 정규화(Lasso)를 적용한 다항 회귀 모델을 학습시켜 볼게요. 여기서 alpha는 정규화의 강도를 의미하는 파라미터예요. 값이 커질수록 불필요한 피처의 계수가 0이 되면서 모델이 더 단순해집니다. max_iter는 쉽게 말하면 모델이 손실을 줄이기 위해 반복적으로 값을 조정하는 과정을 최대 몇 번까지 수행할지 정하는 값이라고 보면 돼요.

In [27]:
lasso = Lasso(alpha=0.001, max_iter=2000)
lasso.fit(X_train_scaled, y_train)

train_pred = lasso.predict(X_train_scaled)
test_pred = lasso.predict(X_test_scaled)

print("\nPolynomial Regression + Lasso (L1)")
print("Train MSE:", mean_squared_error(y_train, train_pred))
print("Test  MSE:", mean_squared_error(y_test, test_pred))


Polynomial Regression + Lasso (L1)
Train MSE: 0.0033783531862925866
Test  MSE: 0.003502156857770857


처음에 비해 training MSE가 더 커지긴 했지만 test MSE가 많이 줄었어요. 두 값의 차이가 크지 않은 걸 보면 과적합이 크게 완화된 걸 알 수 있습니다.

다음으로 L2 정규화(Ridge)를 적용해 보겠습니다. Ridge에서도 alpha는 정규화 강도를 의미하고, 값이 커질수록 계수가 전체적으로 작아지면서 과적합이 완화돼요.

In [28]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

train_pred = ridge.predict(X_train_scaled)
test_pred = ridge.predict(X_test_scaled)

print("\nPolynomial Regression + Ridge (L2)")
print("Train MSE:", mean_squared_error(y_train, train_pred))
print("Test  MSE:", mean_squared_error(y_test, test_pred))


Polynomial Regression + Ridge (L2)
Train MSE: 0.003375607884810807
Test  MSE: 0.003500434708126819


이번에도 역시 training MSE와 test MSE 사이의 차이가 작아 과적합이 완화된 걸 확인할 수 있어요.

정리해 보면, 높은 차수의 다항 회귀는 과소적합을 막는 데는 효과적이지만 과적합이 발생하기 쉽습니다. 여기에 Ridge나 Lasso 같은 정규화를 적용하면, training 성능은 조금 떨어지더라도 test 성능이 개선되면서 더 좋은 모델을 만들 수 있습니다.